# 08I: Development cross-view identity refinement

This notebook tests whether foreground DINO patch pooling improves object association across uploaded views. It uses the existing 100-scene development screen and does not change the 2,000-scene split.

DINOv2-small, LocateAnything proposals, SAM 2.1 masks and all existing scorer outputs remain fixed. The notebook reuses the locked 08G global embeddings and only computes the foreground patch descriptors that were not retained after scoring.

Notebook file: `08i-development-identity-refinement.ipynb`.

## Saved-output evidence copy

This notebook records the completed run for repository review. Its code and markdown match the clean runnable notebook. Private scene visuals and identifier-bearing output were removed. Safe aggregate tables and charts were retained.

Run stage: `08I-screen100`  
Clean notebook SHA-256: `e955c4a7a1f619f6ccc8d050293fd4d3a11726507cda72b5cc9b0f19198e8e1a`  
Downloaded saved-output SHA-256: `bca913030711c7fc5dc4d49e365d23742087bde022055513f5fe5145a3115f39`


## Study boundary

The automatic route is the main route because the web application will not receive reference boxes or masks. The notebook compares two, three and five selected views. It keeps the current global DINO identity method as the baseline and adds local and hybrid descriptors built from foreground patch tokens.

Lock prediction-only identity candidates before any development correspondence is opened. Only then open development correspondence for evaluation. Labels, categories, asset groups and source object identities cannot enter descriptor extraction or matching.

No arbitrary accuracy gate is used. Report actual identity precision, identity recall, identity F1, average precision, top-1 accuracy, recall at two and subgroup results. The highest development row is a candidate for review, not an automatic promotion or an untouched test result.

## Paths and runtime settings

Attach four private resources: the identity-refinement source package, the existing scoring model-input package, the completed 08G screen output and the matching development evaluator. A recovery archive is optional for a fresh run and required only for an intentional continuation.

In [1]:
from pathlib import Path, PurePosixPath
from hashlib import sha256
from zipfile import ZipFile
import json
import os
import shutil
import stat
import subprocess
import sys

INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')
PACKAGE_ROOT = WORKING_ROOT / 'identity-packages'
PREDICTION_OUTPUT = WORKING_ROOT / 'identity-refinement-predictions'
EVALUATION_OUTPUT = WORKING_ROOT / 'identity-refinement-evaluation'
PYTHON = Path(sys.executable)
EXPECT_RECOVERY = False

mount_roots = []
for pattern in ('*', 'datasets/*/*', 'datasets/*/*/versions/*', 'notebooks/*/*', 'notebooks/*/*/versions/*'):
    mount_roots.extend(path.resolve() for path in INPUT_ROOT.glob(pattern) if path.is_dir())
mount_roots = sorted(set(mount_roots))
source_mounts = [path for path in mount_roots if any(part.startswith('vpi-identity-refinement-source') for part in path.parts[-4:])]
source_markers = []
for source_mount in source_mounts:
    source_markers.extend(path for path in source_mount.rglob('identity_refinement.py') if path.is_file())
source_candidates = [path.parents[3] for path in sorted(set(source_markers)) if (path.parents[3] / 'scripts/run_identity_refinement.py').is_file()]
if not source_candidates:
    archives = []
    for source_mount in source_mounts:
        archives.extend(path for path in source_mount.rglob('identity-refinement-source-*.zip') if path.is_file())
    archives = sorted(set(archives))
    if len(archives) != 1:
        raise ValueError(f'Expected one identity-refinement source archive, found {len(archives)}')
    extraction = WORKING_ROOT / 'identity-source'
    temporary = extraction.with_name(extraction.name + '.incomplete')
    if extraction.exists() or temporary.exists():
        raise FileExistsError('Source extraction destination is not empty')
    temporary.mkdir(parents=True)
    with ZipFile(archives[0]) as handle:
        entries = handle.infolist()
        names = [entry.filename for entry in entries]
        if not entries or len(names) != len(set(names)) or handle.testzip() is not None:
            raise ValueError('Source archive is empty, duplicated or corrupt')
        for entry in entries:
            relative = PurePosixPath(entry.filename)
            mode = entry.external_attr >> 16
            if entry.is_dir() or relative.is_absolute() or any(part in ('', '.', '..') for part in relative.parts) or stat.S_ISLNK(mode):
                raise ValueError('Source archive contains an unsafe path')
            destination = temporary.joinpath(*relative.parts)
            destination.parent.mkdir(parents=True, exist_ok=True)
            with handle.open(entry) as source_stream, destination.open('xb') as target_stream:
                shutil.copyfileobj(source_stream, target_stream, length=1024 * 1024)
    temporary.replace(extraction)
    source_candidates = [extraction / 'identity-refinement']
if len(source_candidates) != 1:
    raise ValueError(f'Expected one identity-refinement source root, found {len(source_candidates)}')
SOURCE = source_candidates[0]
sys.path.insert(0, str(SOURCE / 'scripts'))
from kaggle_mounts import find_prediction, find_recovery, materialize_sealed_resource
from scoring_comparison_runtime import run_bounded, tracking_credentials

## Runtime preparation

Install the pinned add-ons into a run-local package directory. Retain Kaggle's CUDA-compatible PyTorch build. Package installation, tests and input validation are bounded and finish before model loading.

In [2]:
PACKAGE_ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run([
    str(PYTHON), '-m', 'pip', 'install', '--disable-pip-version-check',
    '--retries', '1', '--timeout', '60', '--only-binary=:all:',
    '--upgrade', '--target', str(PACKAGE_ROOT),
    '-r', str(SOURCE / 'requirements/identity-refinement.txt'),
], check=True, timeout=600)
os.environ['PYTHONPATH'] = os.pathsep.join([
    str(SOURCE / 'src'), str(PACKAGE_ROOT), os.environ.get('PYTHONPATH', '')
]).rstrip(os.pathsep)
sys.path.insert(0, str(PACKAGE_ROOT))
RUNNER = SOURCE / 'scripts/run_identity_refinement.py'
test_env = dict(os.environ, PYTHONDONTWRITEBYTECODE='1', MPLCONFIGDIR=str(WORKING_ROOT / 'matplotlib'))
subprocess.run([
    str(PYTHON), '-m', 'pytest', '-p', 'no:cacheprovider',
    str(SOURCE / 'tests/test_identity_patch_matching.py'),
    str(SOURCE / 'tests/test_identity_refinement_contract.py'), '-q',
], check=True, cwd=WORKING_ROOT, env=test_env, timeout=300)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.0/125.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.8/780.8 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 5.0.0 requires fsspec[http]<=2026.4.0,>=2023.1.0, but you have fsspec 2026.7.0 which is incompatible.
ydata-profiling 4.18.4 requires matplotlib<=3.10,>=3.5, but you have matplotlib 3.10.8 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator

........                                                                 [100%]
8 passed in 1.64s


CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-p', 'no:cacheprovider', '/kaggle/input/datasets/esreekarreddy/vpi-identity-refinement-source/identity-refinement/tests/test_identity_patch_matching.py', '/kaggle/input/datasets/esreekarreddy/vpi-identity-refinement-source/identity-refinement/tests/test_identity_refinement_contract.py', '-q'], returncode=0)

## Locked input discovery

Materialise the existing model-only package and locate the completed 08G screen output by its lock binding. The evaluator resource is not opened during this phase.

In [3]:
MODEL_INPUT = materialize_sealed_resource(
    INPUT_ROOT,
    kind='model-input',
    archive_name='scoring-comparison-model-input.zip',
    archive_sha256='6d6b45aa211ea595524729123870307ee5295ada48e6fd8f1898935f477b6901',
    destination=WORKING_ROOT / 'identity-model-resource',
    expected_lock_sha256='9baa53454d2482caf8240e7a8b591a6e9e9646dd160c5b3abdc8bd17265ebda1',
)
model_lock_sha256 = sha256((MODEL_INPUT / 'artifact_lock.json').read_bytes()).hexdigest()
SCORING = find_prediction(INPUT_ROOT, expected_input_lock_sha256=model_lock_sha256)
if not (SCORING / 'feature_lock.json').is_file():
    raise FileNotFoundError('Attach the completed 08G screen output with its feature lock')
run_bounded([
    str(PYTHON), str(RUNNER), 'preflight',
    '--model-input', str(MODEL_INPUT), '--scoring', str(SCORING),
    '--output', str(PREDICTION_OUTPUT),
], WORKING_ROOT / 'identity-input-validation.log', idle_seconds=300, total_seconds=900)

{'scene_count': 100, 'scenes': ['[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected iden

## Checked recovery

Every completed scene has an atomic checkpoint and a matching receipt. The run binding covers inputs, source, dependencies and the fixed protocol. Kaggle recovery must be attached as the original `.zip.bin` bytes together with its manifest. Expanded, incomplete, duplicated or differently bound recovery data is rejected before inference.

In [4]:
# A fresh run must ignore unrelated recovery files mounted by Kaggle.
RECOVERY = find_recovery(INPUT_ROOT) if EXPECT_RECOVERY else None
print(json.dumps({
    'event': 'recovery_discovery',
    'expected': EXPECT_RECOVERY,
    'attached': RECOVERY is not None,
    'archive': RECOVERY[0].name if RECOVERY else None,
}, sort_keys=True))
if EXPECT_RECOVERY and RECOVERY is None:
    raise FileNotFoundError('This continuation run requires one sealed recovery archive and manifest')
recovery_arguments = [] if RECOVERY is None else [
    '--recovery-bundle', str(RECOVERY[0]),
    '--recovery-manifest', str(RECOVERY[1]),
]

{"archive": null, "attached": false, "event": "recovery_discovery", "expected": false}


## Lock prediction-only identity candidates

Use one GPU. The runner downloads the pinned DINOv2-small revision, loads it on CPU, moves it to CUDA and reports each checkpoint. It recomputes foreground patch descriptors for the eight views used across the two, three and five-view comparisons. Global embeddings and all anomaly scores are reused from 08G.

A 15-minute silent limit stops a stalled worker. A four-hour total limit prevents unattended GPU use. These are runtime limits, not accuracy gates. A failure preserves the newest recovery package; it does not justify repeating completed scenes.

In [5]:
run_bounded([
    str(PYTHON), str(RUNNER), 'predict',
    '--model-input', str(MODEL_INPUT), '--scoring', str(SCORING),
    '--output', str(PREDICTION_OUTPUT), *recovery_arguments,
], WORKING_ROOT / 'identity-prediction.log', idle_seconds=900, total_seconds=14400)
run_bounded([
    str(PYTHON), str(RUNNER), 'verify',
    '--model-input', str(MODEL_INPUT), '--scoring', str(SCORING),
    '--output', str(PREDICTION_OUTPUT),
], WORKING_ROOT / 'identity-prediction-validation.log', idle_seconds=300, total_seconds=600)
display(json.loads((PREDICTION_OUTPUT / 'run_status.json').read_text()))
display(json.loads((PREDICTION_OUTPUT / 'progress.json').read_text()))

{'event': 'runtime_ready', 'gpu': 'Tesla T4', 'visible_gpus': 2, 'devices_used': 1, 'output_free_bytes': 20351041536}
{"event": "model_load_start", "revision": "ed25f3a31f01632728cabb09d1542f84ab7b0056"}
{'event': 'model_download_start', 'revision': 'ed25f3a31f01632728cabb09d1542f84ab7b0056'}

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Fetching 6 files: 100%|██████████| 6/6 [00:06<00:00,  1.06s/it]
{'event': 'model_download_complete', 'snapshot': 'ed25f3a31f01632728cabb09d1542f84ab7b0056'}

Loading weights: 100%|██████████| 223/223 [00:00<00:00, 2874.25it/s]
{'event': 'model_cpu_ready'}
{'event': 'model_cuda_ready', 'allocated_bytes': 88226304}
{"event": "model_ready"}
{"event": "scene_complete", "scene_number": 1, "total": 100}
{"event": "scene_complete", "scene_number": 2, "total": 100}
{"event": "scene_complete", "scene_number": 3, "t

{'scene_count': 100, 'status': 'complete'}

{'complete_scenes': 100, 'expected_scenes': 100}

## Only then open development correspondence

The prediction lock now fixes every descriptor and match before labels are read. The evaluator checks true cross-view identity and the downstream odd-one-out ranking. The automatic route remains the denominator even when proposals are missing or tracks are mixed.

In [6]:
EVALUATOR = materialize_sealed_resource(
    INPUT_ROOT,
    kind='evaluator-only',
    archive_name='scoring-comparison-evaluator-only.zip',
    archive_sha256='86ac512005af5fdceb64c0fb666998832b8ee08ef96d1fccca23c6e35f420c12',
    destination=WORKING_ROOT / 'identity-evaluator-resource',
    expected_lock_sha256='8012cd20c63b3788f861f2ba991b3181a0c1666f26e176beeee370228c8500c3',
)
run_bounded([
    str(PYTHON), str(RUNNER), 'evaluate',
    '--model-input', str(MODEL_INPUT), '--scoring', str(SCORING),
    '--predictions', str(PREDICTION_OUTPUT), '--evaluator', str(EVALUATOR),
    '--output', str(EVALUATION_OUTPUT),
], WORKING_ROOT / 'identity-evaluation.log', idle_seconds=900, total_seconds=7200)
run_bounded([
    str(PYTHON), str(RUNNER), 'verify',
    '--model-input', str(MODEL_INPUT), '--scoring', str(SCORING),
    '--output', str(EVALUATION_OUTPUT),
], WORKING_ROOT / 'identity-evaluation-validation.log', idle_seconds=300, total_seconds=600)

{"event": "scene_evaluated", "scene_number": 1, "total": 100}
{"event": "scene_evaluated", "scene_number": 2, "total": 100}
{"event": "scene_evaluated", "scene_number": 3, "total": 100}
{"event": "scene_evaluated", "scene_number": 4, "total": 100}
{"event": "scene_evaluated", "scene_number": 5, "total": 100}
{"event": "scene_evaluated", "scene_number": 6, "total": 100}
{"event": "scene_evaluated", "scene_number": 7, "total": 100}
{"event": "scene_evaluated", "scene_number": 8, "total": 100}
{"event": "scene_evaluated", "scene_number": 9, "total": 100}
{"event": "scene_evaluated", "scene_number": 10, "total": 100}
{"event": "scene_evaluated", "scene_number": 11, "total": 100}
{"event": "scene_evaluated", "scene_number": 12, "total": 100}
{"event": "scene_evaluated", "scene_number": 13, "total": 100}
{"event": "scene_evaluated", "scene_number": 14, "total": 100}
{"event": "scene_evaluated", "scene_number": 15, "total": 100}
{"event": "scene_evaluated", "scene_number": 16, "total": 100}
{

## Summary tables

Start with the three-view automatic route and the directed patch scorer. Compare top-1 accuracy with identity link F1 and inspect the paired asset-group intervals. Do not select a method from top-1 alone.

In [7]:
import pandas as pd
for file_name in (
    'primary_candidate_ranking.csv',
    'identity_refinement_summary.csv',
    'paired_asset_intervals.csv',
    'patch_fallback_summary.csv',
    'category_summary.csv',
    'family_summary.csv',
):
    print(file_name)
    display(pd.read_csv(EVALUATION_OUTPUT / 'shareable' / file_name))
display(json.loads((EVALUATION_OUTPUT / 'shareable/development_decision.json').read_text()))

primary_candidate_ranking.csv


,configuration_id,descriptor,unmatched_cost,view_count,method,scenes,asset_groups,ap,top1,recall_at_2,auroc,identity_link_precision,identity_link_recall,identity_link_f1,identity_track_purity,winner_support,winner_purity
0,hybrid50_cov25__u020__v3,hybrid50_cov25,0.20,3,forward_02_cov70,100,27,0.375897,0.31,0.335,0.397954,0.205782,0.113974,0.139612,0.842944,1.44,0.490000
1,global_current__u020__v3,global_current,0.20,3,forward_02_cov70,100,27,0.374627,0.31,0.335,0.403671,0.197740,0.095571,0.121122,0.858556,1.37,0.530000
2,local_cov05__u050__v3,local_cov05,0.50,3,forward_02_cov70,100,27,0.370083,0.30,0.365,0.346250,0.209516,0.206483,0.207789,0.610278,2.69,0.526667
3,hybrid25_cov25__u020__v3,hybrid25_cov25,0.20,3,forward_02_cov70,100,27,0.365397,0.29,0.330,0.393058,0.201177,0.099571,0.126531,0.835500,1.38,0.500000
4,local_cov50__u050__v3,local_cov50,0.50,3,forward_02_cov70,100,27,0.357333,0.28,0.350,0.333155,0.211047,0.207880,0.209220,0.634167,2.67,0.521667
5,local_cov05__u020__v3,local_cov05,0.20,3,forward_02_cov70,100,27,0.351159,0.28,0.330,0.368426,0.205510,0.129021,0.152857,0.807500,1.52,0.458333
6,local_cov50__u035__v3,local_cov50,0.35,3,forward_02_cov70,100,27,0.348083,0.28,0.350,0.332613,0.202532,0.181369,0.190040,0.655278,2.32,0.486667
7,local_cov25__u050__v3,local_cov25,0.50,3,forward_02_cov70,100,27,0.355667,0.27,0.335,0.327321,0.206356,0.203189,0.204529,0.627778,2.67,0.508333
8,hybrid75_cov25__u020__v3,hybrid75_cov25,0.20,3,forward_02_cov70,100,27,0.353560,0.27,0.325,0.386191,0.199722,0.116406,0.140021,0.816111,1.47,0.441667
9,hybrid50_cov25__u050__v3,hybrid50_cov25,0.50,3,forward_02_cov70,100,27,0.347417,0.27,0.335,0.313405,0.200936,0.196094,0.198140,0.625000,2.61,0.518333


identity_refinement_summary.csv


,configuration_id,descriptor,unmatched_cost,view_count,method,scenes,asset_groups,ap,top1,recall_at_2,auroc,identity_link_precision,identity_link_recall,identity_link_f1,identity_track_purity,winner_support,winner_purity
0,global_current__u020__v2,global_current,0.2,2,P1_current,100,27,0.429790,0.33,0.390,0.439851,0.107833,0.059167,0.072437,0.920000,1.20,0.735000
1,global_current__u020__v2,global_current,0.2,2,P2_consensus,100,27,0.416143,0.31,0.390,0.425071,0.107833,0.059167,0.072437,0.920000,1.21,0.685000
2,global_current__u020__v2,global_current,0.2,2,forward_02_cov70,100,27,0.460242,0.39,0.430,0.458291,0.107833,0.059167,0.072437,0.920000,1.23,0.750000
3,global_current__u020__v2,global_current,0.2,2,forward_10_cov0,100,27,0.453444,0.36,0.410,0.453889,0.107833,0.059167,0.072437,0.920000,1.22,0.750000
4,global_current__u020__v3,global_current,0.2,3,P1_current,100,27,0.345850,0.27,0.290,0.371109,0.197740,0.095571,0.121122,0.858556,1.41,0.588333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,local_cov50__u050__v3,local_cov50,0.5,3,forward_10_cov0,100,27,0.339750,0.24,0.310,0.309054,0.211047,0.207880,0.209220,0.634167,2.70,0.516667
248,local_cov50__u050__v5,local_cov50,0.5,5,P1_current,100,27,0.296667,0.19,0.315,0.286333,0.248404,0.241268,0.244491,0.514972,4.35,0.421000
249,local_cov50__u050__v5,local_cov50,0.5,5,P2_consensus,100,27,0.292917,0.18,0.310,0.282878,0.248404,0.241268,0.244491,0.514972,4.32,0.403833
250,local_cov50__u050__v5,local_cov50,0.5,5,forward_02_cov70,100,27,0.343250,0.25,0.400,0.355000,0.248404,0.241268,0.244491,0.514972,4.38,0.429500


paired_asset_intervals.csv


...

category_summary.csv


,configuration_id,descriptor,unmatched_cost,view_count,method,category,ap,top1,identity_link_f1
0,global_current__u020__v2,global_current,0.2,2,P1_current,apple,0.375000,0.250000,0.000000
1,global_current__u020__v2,global_current,0.2,2,P1_current,bottle,0.166667,0.000000,0.083333
2,global_current__u020__v2,global_current,0.2,2,P1_current,bowl,0.833333,0.666667,0.111111
3,global_current__u020__v2,global_current,0.2,2,P1_current,bread,0.431548,0.250000,0.000000
4,global_current__u020__v2,global_current,0.2,2,P1_current,bunny,0.166667,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
6799,local_cov50__u050__v5,local_cov50,0.5,5,forward_10_cov0,monkey,0.000000,0.000000,0.144841
6800,local_cov50__u050__v5,local_cov50,0.5,5,forward_10_cov0,mouse,0.000000,0.000000,0.204074
6801,local_cov50__u050__v5,local_cov50,0.5,5,forward_10_cov0,orange,0.333333,0.333333,0.188333
6802,local_cov50__u050__v5,local_cov50,0.5,5,forward_10_cov0,pig,0.500000,0.333333,0.218333


family_summary.csv


,configuration_id,descriptor,unmatched_cost,view_count,method,families,ap,top1,identity_link_f1
0,global_current__u020__v2,global_current,0.2,2,P1_current,bump,0.550866,0.454545,0.074459
1,global_current__u020__v2,global_current,0.2,2,P1_current,crack,0.425490,0.352941,0.043137
2,global_current__u020__v2,global_current,0.2,2,P1_current,crack+disc,0.541667,0.750000,0.000000
3,global_current__u020__v2,global_current,0.2,2,P1_current,crack+frac,0.495536,0.375000,0.091667
4,global_current__u020__v2,global_current,0.2,2,P1_current,defr,0.166667,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
2767,local_cov50__u050__v5,local_cov50,0.5,5,forward_10_cov0,frac,0.473684,0.315789,0.269185
2768,local_cov50__u050__v5,local_cov50,0.5,5,forward_10_cov0,miss,0.500000,0.375000,0.356944
2769,local_cov50__u050__v5,local_cov50,0.5,5,forward_10_cov0,miss+tran,0.375000,0.500000,0.262500
2770,local_cov50__u050__v5,local_cov50,0.5,5,forward_10_cov0,rota,0.000000,0.000000,0.162656


{'baseline': 'global_current__u035__v3',
 'development_candidate': 'hybrid50_cov25__u020__v3',
 'primary_route': 'automatic masks, predicted identity, three views',
 'primary_scorer': 'forward_02_cov70',
 'promotion': 'No automatic promotion. Review intervals, identity quality, subgroups and galleries.',
 'protected_data_used': False,
 'split_changed': False,
 'status': 'review_required'}

## Graphs and private gallery

The first graph ranks the strongest three-view candidates. The second checks whether an odd-one-out improvement is accompanied by better identity links. The private gallery contains fixed samples, improvements and regressions. Red marks the selected identity, green marks the reference unusual object and blue marks other automatic candidates. A red mask is shown only where the final automatic route produced an object mask.

In [8]:
from IPython.display import Image as NotebookImage, display
for figure in sorted((EVALUATION_OUTPUT / 'shareable/figures').glob('*.png')):
    print(figure.name)
    display(NotebookImage(filename=str(figure)))
gallery = json.loads((EVALUATION_OUTPUT / 'private/gallery_manifest.json').read_text())
for item in gallery['examples']:
    print(item['file'], ':', item['selection_reason'])
    display(NotebookImage(filename=str(EVALUATION_OUTPUT / 'private/figures' / item['file'])))

identity_vs_top1.png


<IPython.core.display.Image object>

primary_top1_comparison.png


<IPython.core.display.Image object>

identity_example_01.png : improvement


...

<IPython.core.display.Image object>

identity_example_11.png : fixed_hash_sample


<IPython.core.display.Image object>

identity_example_12.png : fixed_hash_sample


<IPython.core.display.Image object>

## Experiment tracking

ClearML runs only after the scientific outputs are sealed. It receives aggregate tables, decision records, summary graphs, metrics and checksums. It does not receive private galleries, labels, per-scene rows, raw images or credentials. A tracking failure can be retried without rerunning DINO.

In [9]:
if tracking_credentials():
    try:
        subprocess.run([
            str(PYTHON), '-m', 'pip', 'install', '--retries', '1', '--timeout', '30',
            '--only-binary=:all:', '--upgrade', '--target', str(PACKAGE_ROOT),
            '-r', str(SOURCE / 'requirements/identity-refinement-tracking.txt'),
        ], check=True, timeout=180)
        subprocess.run([
            str(PYTHON), str(SOURCE / 'scripts/log_identity_refinement.py'),
            '--evidence', str(EVALUATION_OUTPUT),
        ], check=True, timeout=180)
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
        print('ClearML upload is not verified. Preserve the outputs and retry only the tracking command.')
else:
    print('ClearML secrets are unavailable. Preserve the outputs and enable the existing notebook secrets.')

ClearML secrets are unavailable. Preserve the outputs and enable the existing notebook secrets.


## Evidence export and review

Download the executed notebook, three log files, both complete output directories and the recovery archive with its manifest. Verify lock files after download. Keep private gallery images, per-scene metrics, model inputs and recovery data outside GitHub. A later repository package should contain the clean notebook, source, tests, safe tables, summary figures, sanitized executed notebook and verified run receipts.

Review successes and failures before freezing a method. If no local or hybrid descriptor improves both downstream ranking and identity quality, retain the current method and keep optional human correction after the automatic result.

In [10]:
print('Prediction evidence:', PREDICTION_OUTPUT)
print('Evaluation evidence:', EVALUATION_OUTPUT)
print('Recovery archive:', PREDICTION_OUTPUT.parent / f'{PREDICTION_OUTPUT.name}-recovery.zip')
print('The development decision remains pending manual review.')

Prediction evidence: /kaggle/working/identity-refinement-predictions
Evaluation evidence: /kaggle/working/identity-refinement-evaluation
Recovery archive: /kaggle/working/identity-refinement-predictions-recovery.zip
The development decision remains pending manual review.


## Working-directory cleanup

Remove only disposable dependency, extraction and plotting folders after outputs and optional tracking have finished. Prediction checkpoints, locks, logs, tables, graphs and galleries remain available for download.

In [11]:
for path in (
    PACKAGE_ROOT,
    WORKING_ROOT / 'matplotlib',
    WORKING_ROOT / 'identity-source',
    WORKING_ROOT / 'identity-model-resource',
    WORKING_ROOT / 'identity-evaluator-resource',
):
    if path.exists():
        shutil.rmtree(path)
print('Disposable directories removed. Scientific evidence is unchanged.')

Disposable directories removed. Scientific evidence is unchanged.
